# Screen Images Classification Baselines
In this notebook, we run kNN and Random Forest on the screen images as baselines.

In [1]:
import sys
import os

import torch
from torchvision.transforms import v2
from screen_images import CustomEnricoDataset, get_allowed_classes, make_screen_base_transform
from classification_baseline import extract_features, train_evaluate_knn, train_evaluate_random_forest

## Data Loading and Preprocessing
We use a smaller image size for classical ML algorithms to keep the feature dimension manageable.

In [2]:
# Data configuration
BATCH_SIZE = 64
ENRICO_CLASSES = len(get_allowed_classes())

# We resize the image for basic machine learning models so that the flattened feature vector 
# isnt overly huge (e.g., 60x40 means 60*40*3 = 7200 features)
# We use make_screen_base_transform proportional to 300x200
screen_preprocess = make_screen_base_transform(resize=(60, 40))

# Use the data path that you have configured in your previous notebooks
root_dir = "/kaggle/input/datasets/nazariyyuchnovskiy/enricoscreenshotsandwireframes"

try:
    print(f"Loading data from: {root_dir}")
    train_dataset, val_dataset, test_dataset = CustomEnricoDataset.create_splits(
        root=root_dir,
        val_size=0.1,
        test_size=0.1,
        use_wireframes=False,
        train_transform=screen_preprocess,
        eval_transform=screen_preprocess
    )

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    print("Data splits and loaders created successfully!")
except Exception as e:
    print(f"Error loading dataset: {e}")


Loading data from: /kaggle/input/datasets/nazariyyuchnovskiy/enricoscreenshotsandwireframes
Data splits and loaders created successfully!


## Feature Extraction
Flatten the multidimensional tensors into 1D vectors per image.

In [3]:
# Extract features (flatten images into 1D vectors for scikit-learn models)
print("Extracting features for training data...")
X_train, y_train = extract_features(train_loader, flatten=True)

print("Extracting features for testing data...")
X_test, y_test = extract_features(test_loader, flatten=True)

print(f"Train data shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test data shape: X={X_test.shape}, y={y_test.shape}")


Extracting features for training data...
Extracting features for testing data...
Train data shape: X=(979, 7200), y=(979,)
Test data shape: X=(123, 7200), y=(123,)


## k-Nearest Neighbors (kNN) Classifier

In [4]:
# kNN Baseline
# Setting n_jobs=-1 to use all available CPU cores for faster computation
knn_model, knn_preds = train_evaluate_knn(X_train, y_train, X_test, y_test, n_neighbors=15, n_jobs=-1)


Training kNN with n_neighbors=15...
Evaluating kNN...
kNN Results -> Accuracy: 0.2764 | Precision: 0.2627 | Recall: 0.2764 | F1: 0.2251

Classification Report (kNN):
              precision    recall  f1-score   support

           0       0.10      0.14      0.12         7
           1       0.00      0.00      0.00        11
           2       0.25      0.07      0.11        15
           3       0.31      0.78      0.44        27
           4       0.00      0.00      0.00        14
           5       0.57      0.50      0.53         8
           6       0.40      0.33      0.36         6
           7       1.00      0.20      0.33         5
           8       0.00      0.00      0.00         9
           9       0.00      0.00      0.00         4
          10       0.44      0.24      0.31        17

    accuracy                           0.28       123
   macro avg       0.28      0.21      0.20       123
weighted avg       0.26      0.28      0.23       123



## Random Forest Classifier

In [5]:
# Random Forest Baseline
# Using 100 estimators as a solid baseline
rf_model, rf_preds = train_evaluate_random_forest(X_train, y_train, X_test, y_test, n_estimators=100, n_jobs=-1, random_state=42)


Training Random Forest with n_estimators=100...
Evaluating Random Forest...
Random Forest Results -> Accuracy: 0.4715 | Precision: 0.4668 | Recall: 0.4715 | F1: 0.4370

Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       0.57      0.57      0.57         7
           1       0.25      0.09      0.13        11
           2       0.44      0.47      0.45        15
           3       0.48      0.85      0.61        27
           4       0.37      0.50      0.42        14
           5       0.83      0.62      0.71         8
           6       0.33      0.33      0.33         6
           7       1.00      0.40      0.57         5
           8       0.50      0.11      0.18         9
           9       0.00      0.00      0.00         4
          10       0.46      0.35      0.40        17

    accuracy                           0.47       123
   macro avg       0.48      0.39      0.40       123
weighted avg       0.47      0.47 